# 🌿 HUẤN LUYỆN & CHIA SEED ĐỐI CHỨNG (SEED 42, SEED 99, SEED 100)
**Đề tài:** Nhận diện 90 loài cây kiểng lá (*FloraBotanic AI*)
**Mục tiêu:** Chia lại tập dữ liệu theo tỷ lệ chuẩn `80% Train - 10% Val - 10% Test` với **Random Seed tùy chọn** (`42`, `99`, hoặc `100`), huấn luyện mô hình MobileNetV2 và tự động xuất bảng so sánh đối chứng tính tái lập (Reproducibility) theo yêu cầu của Giảng viên hướng dẫn.

> 💡 **Hướng dẫn chạy:**
> 1. Chỉ cần thay đổi biến `RANDOM_SEED` ở **BƯỚC 1** (chọn `42`, `99` hoặc `100`).
> 2. Nhấn `Runtime -> Run all` (hoặc chạy lần lượt từng Cell từ trên xuống dưới).
> 3. Dữ liệu và kết quả mỗi Seed sẽ được lưu ở thư mục riêng, không bị ghi đè lẫn nhau!

In [ ]:
# ==============================================================================
# === BƯỚC 1: CẤU HÌNH RANDOM SEED & KẾT NỐI GOOGLE DRIVE                    ===
# ==============================================================================
# 👉 BẠN CHỈ CẦN THAY ĐỔI CON SỐ NÀY KHI CHẠY TỪNG LẦN:
RANDOM_SEED = 42   # Lần 1: 42 | Lần 2: 99 | Lần 3: 100

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import shutil
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
import tensorflow as tf
from tensorflow import keras

# 1. Cố định toàn diện Random Seed cho toàn bộ hệ thống
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)

print(f'✅ ĐÃ THIẾT LẬP TOÀN BỘ HỆ THỐNG VỚI RANDOM SEED = {RANDOM_SEED}')
print('TensorFlow version:', tf.__version__)
print('GPU Devices:', tf.config.list_physical_devices('GPU'))

# 2. Cấu hình thư mục lưu trữ riêng cho từng Seed (không sợ bị đè model cũ!)
PROJECT = Path('/content/drive/MyDrive/KiengLa_AI')
ZIP_PATH = PROJECT / 'merged_v1/merged_90_split.zip'
RAW_EXTRACT_DIR = Path('/content/merged_90_raw')

# Thư mục dữ liệu đã chia và mô hình lưu riêng theo Seed:
DATA_DIR = Path(f'/content/merged_90_split_seed_{RANDOM_SEED}')
SAVE_ROOT = PROJECT / f'train_90_lop_seed_{RANDOM_SEED}'
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

print(f'📁 Thư mục dữ liệu chia: {DATA_DIR}')
print(f'📁 Thư mục lưu Model   : {SAVE_ROOT}')


In [ ]:
# ==============================================================================
# === BƯỚC 2: GIẢI NÉN VÀ CHIA TẬP 80-10-10 THEO RANDOM SEED NÀY             ===
# ==============================================================================
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Giải nén bộ dữ liệu gốc nếu chưa giải nén
if not (RAW_EXTRACT_DIR / 'train').exists():
    assert ZIP_PATH.is_file(), f'Không tìm thấy file: {ZIP_PATH}'
    print('Đang giải nén bộ dữ liệu gốc...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as arc:
        arc.extractall(RAW_EXTRACT_DIR)
    print('Giải nén thành công!')

# Kiểm tra nếu chưa chia cho Seed này thì tiến hành phân chia Stratified Split
flag_file = DATA_DIR / f'_split_seed_{RANDOM_SEED}_ok'
if not flag_file.exists():
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)

    class_folders = sorted([d.name for d in (RAW_EXTRACT_DIR / 'train').iterdir() if d.is_dir()])
    assert len(class_folders) == 90, f'Số lớp không đúng 90: {len(class_folders)}'

    rng = random.Random(RANDOM_SEED)  # Bộ sinh ngẫu nhiên cố định theo Seed
    split_rows = []

    for cls_name in class_folders:
        all_imgs = []
        for old_split in ['train', 'val', 'test']:
            cls_dir = RAW_EXTRACT_DIR / old_split / cls_name
            if cls_dir.is_dir():
                all_imgs.extend([
                    p for p in cls_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in EXTENSIONS
                ])

        # Sắp xếp cố định rồi shuffle theo Seed
        all_imgs = sorted(all_imgs, key=lambda p: p.name)
        rng.shuffle(all_imgs)

        n = len(all_imgs)
        n_train = int(n * 0.80)
        n_val = int(n * 0.90) - n_train
        if n_val == 0 and n >= 3:
            n_val = 1

        splits = {
            'train': all_imgs[:n_train],
            'val': all_imgs[n_train:n_train + n_val],
            'test': all_imgs[n_train + n_val:]
        }

        for split_name, paths in splits.items():
            dest_dir = DATA_DIR / split_name / cls_name
            dest_dir.mkdir(parents=True, exist_ok=True)
            for src_path in paths:
                dst_path = dest_dir / src_path.name
                try:
                    os.link(src_path, dst_path)
                except OSError:
                    shutil.copy2(src_path, dst_path)

        split_rows.append({
            'label': cls_name,
            'train': len(splits['train']),
            'val': len(splits['val']),
            'test': len(splits['test']),
            'total': n
        })

    split_df = pd.DataFrame(split_rows)
    split_df.to_csv(SAVE_ROOT / f'split_counts_seed_{RANDOM_SEED}.csv', index=False, encoding='utf-8-sig')
    flag_file.touch()
    print(f'✅ Đã chia xong 80-10-10 cho Seed {RANDOM_SEED}!')
else:
    split_df = pd.read_csv(SAVE_ROOT / f'split_counts_seed_{RANDOM_SEED}.csv')
    print(f'✅ Đã tìm thấy dữ liệu đã chia trước đó cho Seed {RANDOM_SEED}.')

tot_train = int(split_df['train'].sum())
tot_val = int(split_df['val'].sum())
tot_test = int(split_df['test'].sum())
print(f'Phân bổ: Train = {tot_train:,} | Val = {tot_val:,} | Test = {tot_test:,}')


In [ ]:
# ==============================================================================
# === BƯỚC 3: TẠO DATASET TENSORFLOW & TÍNH BALANCED CLASS WEIGHTS           ===
# ==============================================================================
from sklearn.utils.class_weight import compute_class_weight

def load_dataset(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        str(DATA_DIR / split),
        image_size=(224, 224),
        interpolation='nearest',
        batch_size=32,
        label_mode='int',
        shuffle=shuffle,
        seed=RANDOM_SEED
    )

train_ds_raw = load_dataset('train', True)
val_ds_raw = load_dataset('val', False)
test_ds_raw = load_dataset('test', False)

class_names = train_ds_raw.class_names
assert len(class_names) == 90

y_train_all = np.concatenate([y.numpy() for _, y in train_ds_raw], axis=0)
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(class_names)),
    y=y_train_all
)
class_weight_dict = {i: float(w) for i, w in enumerate(weights)}

train_ds = train_ds_raw.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds_raw.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds_raw.prefetch(tf.data.AUTOTUNE)

HEAD_DIR = SAVE_ROOT / 'head'
FINE_DIR = SAVE_ROOT / 'finetune'
HEAD_DIR.mkdir(parents=True, exist_ok=True)
FINE_DIR.mkdir(parents=True, exist_ok=True)

(SAVE_ROOT / 'class_names.json').write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
(HEAD_DIR / 'class_names.json').write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
(FINE_DIR / 'class_names.json').write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'✅ Pipeline đã sẵn sàng cho 90 lớp với {len(y_train_all):,} ảnh Train.')


In [ ]:
# ==============================================================================
# === BƯỚC 4: HUẤN LUYỆN GIAI ĐOẠN 1 (TRANSFER LEARNING - ĐÓNG BĂNG BACKBONE) ===
# ==============================================================================
import math
from tensorflow.keras import layers

HEAD_STATE_PATH = HEAD_DIR / 'progress.json'
HEAD_BEST_PATH = HEAD_DIR / 'best_model.keras'

class SaveAndCheck(keras.callbacks.Callback):
    def __init__(self, save_dir, state_path, best_path, patience=5, min_delta=0.001):
        super().__init__()
        self.save_dir = save_dir
        self.state_path = state_path
        self.best_path = best_path
        self.patience = patience
        self.min_delta = min_delta
        if self.state_path.exists():
            self.state = json.loads(self.state_path.read_text(encoding='utf-8'))
        else:
            self.state = {'next_epoch': 0, 'best_loss': None, 'reference_loss': None, 'wait': 0, 'stopped': False}

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = float(logs['val_loss'])
        if not math.isfinite(loss):
            self.model.stop_training = True
            return
        if self.state['best_loss'] is None or loss < self.state['best_loss']:
            temp = self.save_dir / 'best_temp.keras'
            self.model.save(str(temp))
            os.replace(temp, self.best_path)
            self.state['best_loss'] = loss
        ref = self.state['reference_loss']
        if ref is None or (ref - loss) > self.min_delta:
            self.state['reference_loss'] = loss
            self.state['wait'] = 0
        else:
            self.state['wait'] += 1
            if self.state['wait'] >= self.patience:
                print(f'Dừng sớm (Early Stop) tại epoch {epoch+1}')
                self.state['stopped'] = True
                self.model.stop_training = True
        self.state['next_epoch'] = epoch + 1
        self.state_path.write_text(json.dumps(self.state, indent=2), encoding='utf-8')

save_head_cb = SaveAndCheck(HEAD_DIR, HEAD_STATE_PATH, HEAD_BEST_PATH, patience=5)

if save_head_cb.state['stopped']:
    print('✅ Giai đoạn 1 (Head) đã hoàn thành trước đó.')
else:
    base_model = keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = layers.RandomFlip('horizontal', seed=RANDOM_SEED)(inputs)
    x = layers.RandomRotation(0.08, seed=RANDOM_SEED)(x)
    x = layers.RandomZoom(0.10, seed=RANDOM_SEED)(x)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3, seed=RANDOM_SEED)(x)
    outputs = layers.Dense(len(class_names), activation='softmax', name='classifier_90')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy', keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy')]
    )
    print(f'🚀 Bắt đầu huấn luyện Giai đoạn 1 cho Seed {RANDOM_SEED}...')
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        class_weight=class_weight_dict,
        callbacks=[save_head_cb, keras.callbacks.CSVLogger(str(HEAD_DIR / 'history.csv'))]
    )


In [ ]:
# ==============================================================================
# === BƯỚC 5: HUẤN LUYỆN GIAI ĐOẠN 2 (FINE-TUNING 30 LỚP CUỐI MOBILENETV2)     ===
# ==============================================================================
FINE_STATE_PATH = FINE_DIR / 'progress.json'
FINE_BEST_PATH = FINE_DIR / 'best_model.keras'

save_fine_cb = SaveAndCheck(FINE_DIR, FINE_STATE_PATH, FINE_BEST_PATH, patience=5)

if save_fine_cb.state['stopped']:
    print('✅ Giai đoạn 2 (Fine-tuning) đã hoàn thành trước đó.')
else:
    model = keras.models.load_model(str(HEAD_BEST_PATH))
    base_model = next(
        layer for layer in model.layers
        if isinstance(layer, keras.Model) and 'mobilenet' in layer.name.lower()
    )
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        layer.trainable = not isinstance(layer, layers.BatchNormalization)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy', keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy')]
    )
    print(f'🚀 Bắt đầu Fine-tuning cho Seed {RANDOM_SEED}...')
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        class_weight=class_weight_dict,
        callbacks=[save_fine_cb, keras.callbacks.CSVLogger(str(FINE_DIR / 'history.csv'))]
    )


In [ ]:
# ==============================================================================
# === BƯỚC 6: ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP & LƯU KẾT QUẢ ĐỐI CHỨNG CỦA SEED ===
# ==============================================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

EVAL_DIR = SAVE_ROOT / f'evaluation_report_seed_{RANDOM_SEED}'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

best_model = keras.models.load_model(str(FINE_BEST_PATH), compile=False)

true_batches, score_batches = [], []
for images, labels in test_ds:
    scores = best_model(images, training=False).numpy()
    true_batches.append(labels.numpy())
    score_batches.append(scores)

y_true = np.concatenate(true_batches)
scores_all = np.concatenate(score_batches)
y_pred = scores_all.argmax(axis=1)

top1_acc = float(accuracy_score(y_true, y_pred))
top3_acc = float(np.mean(np.any(np.argsort(scores_all, axis=1)[:, -3:] == y_true[:, None], axis=1)))
macro_prec = float(precision_score(y_true, y_pred, average='macro', zero_division=0))
macro_rec = float(recall_score(y_true, y_pred, average='macro', zero_division=0))
macro_f1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
weighted_prec = float(precision_score(y_true, y_pred, average='weighted', zero_division=0))
weighted_rec = float(recall_score(y_true, y_pred, average='weighted', zero_division=0))
weighted_f1 = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))

metrics_dict = {
    'random_seed': RANDOM_SEED,
    'split_ratio': '80-10-10',
    'test_images': int(len(y_true)),
    'correct_predictions': int(np.sum(y_true == y_pred)),
    'incorrect_predictions': int(np.sum(y_true != y_pred)),
    'top1_accuracy': top1_acc,
    'top3_accuracy': top3_acc,
    'macro_precision': macro_prec,
    'macro_recall': macro_rec,
    'macro_f1': macro_f1,
    'weighted_precision': weighted_prec,
    'weighted_recall': weighted_rec,
    'weighted_f1': weighted_f1
}

# Lưu metrics ra Drive
(EVAL_DIR / 'metrics.json').write_text(json.dumps(metrics_dict, indent=2), encoding='utf-8')
cm = confusion_matrix(y_true, y_pred, labels=np.arange(90))
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(EVAL_DIR / 'confusion_matrix.csv')

print('=' * 68)
print(f'🎯 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH VỚI RANDOM SEED = {RANDOM_SEED}')
print('=' * 68)
print(f'  • Top-1 Accuracy    : {top1_acc * 100:.2f}%')
print(f'  • Top-3 Accuracy    : {top3_acc * 100:.2f}%')
print(f'  • Weighted Precision: {weighted_prec * 100:.2f}%')
print(f'  • Weighted Recall   : {weighted_rec * 100:.2f}%')
print(f'  • Weighted F1-Score : {weighted_f1 * 100:.2f}%')
print('=' * 68)


In [ ]:
# ==============================================================================
# === BƯỚC 7: TỔNG HỢP MEAN ± STD VÀ XUẤT BẢNG LATEX CHO BÁO CÁO / OVERLEAF ===
# ==============================================================================
# Cell này tính toán Giá trị trung bình ± Độ lệch chuẩn (Mean ± Std)
# và tự động sinh mã bảng LaTeX (chuẩn Overleaf) y hệt như bài báo khoa học!

import numpy as np
import pandas as pd
import json
from pathlib import Path

seeds = [42, 99, 100]
metrics_data = {}
reports_data = {}

for s in seeds:
    m_file = PROJECT / f"train_90_lop_seed_{s}" / f"evaluation_report_seed_{s}" / "metrics.json"
    r_file = PROJECT / f"train_90_lop_seed_{s}" / f"evaluation_report_seed_{s}" / "classification_report.csv"
    if m_file.exists():
        metrics_data[s] = json.loads(m_file.read_text(encoding="utf-8"))
    if r_file.exists():
        reports_data[s] = pd.read_csv(r_file, index_col=0)

found_seeds = list(metrics_data.keys())
print(f"✅ Đã tìm thấy kết quả của {len(found_seeds)} seeds: {found_seeds}")

if len(found_seeds) == 0:
    print("⚠️ Chưa tìm thấy kết quả. Hãy chạy Bước 1-6 trước!")
else:
    # 1. TÍNH TOÁN MEAN ± STD CHO CÁC CHỈ SỐ TOÀN CỤC (OVERALL METRICS)
    def calc_mean_std(val_list, precision=3, to_latex=False):
        arr = np.array(val_list, dtype=float)
        m = np.mean(arr)
        s = np.std(arr, ddof=1) if len(arr) > 1 else 0.0
        if to_latex:
            return f"${m:.{precision}f} \\pm {s:.{precision}f}$"
        return f"{m:.{precision}f} ± {s:.{precision}f}"

    acc_list = [metrics_data[s]["top1_accuracy"] for s in found_seeds]
    top3_list = [metrics_data[s]["top3_accuracy"] for s in found_seeds]
    prec_list = [metrics_data[s]["weighted_precision"] for s in found_seeds]
    rec_list = [metrics_data[s]["weighted_recall"] for s in found_seeds]
    f1_list = [metrics_data[s]["weighted_f1"] for s in found_seeds]

    summary_df = pd.DataFrame([
        {"Độ đo (Metric)": "Top-1 Accuracy", "Kết quả (Mean ± Std)": calc_mean_std(acc_list), "LaTeX Code": calc_mean_std(acc_list, to_latex=True)},
        {"Độ đo (Metric)": "Top-3 Accuracy", "Kết quả (Mean ± Std)": calc_mean_std(top3_list), "LaTeX Code": calc_mean_std(top3_list, to_latex=True)},
        {"Độ đo (Metric)": "Precision (P)", "Kết quả (Mean ± Std)": calc_mean_std(prec_list), "LaTeX Code": calc_mean_std(prec_list, to_latex=True)},
        {"Độ đo (Metric)": "Recall (R)", "Kết quả (Mean ± Std)": calc_mean_std(rec_list), "LaTeX Code": calc_mean_std(rec_list, to_latex=True)},
        {"Độ đo (Metric)": "F1-Score (F1)", "Kết quả (Mean ± Std)": calc_mean_std(f1_list), "LaTeX Code": calc_mean_std(f1_list, to_latex=True)},
    ])

    print("\n" + "=" * 70)
    print("📊 BẢNG TỔNG HỢP TOÀN CỤC THEO ĐỊNH DẠNG BÀI BÁO KHOA HỌC (MEAN ± STD):")
    print("=" * 70)
    display(summary_df[["Độ đo (Metric)", "Kết quả (Mean ± Std)"]])

    # 2. TÍNH TOÁN MEAN ± STD CHO TỪNG LOÀI CÂY (PER-CLASS P, R, F1)
    if len(reports_data) > 0:
        ref_df = list(reports_data.values())[0]
        class_list = [c for c in ref_df.index if c not in ["accuracy", "macro avg", "weighted avg"]]
        
        per_class_rows = []
        for c in class_list:
            p_vals = [reports_data[s].loc[c, "precision"] for s in found_seeds if c in reports_data[s].index]
            r_vals = [reports_data[s].loc[c, "recall"] for s in found_seeds if c in reports_data[s].index]
            f_vals = [reports_data[s].loc[c, "f1-score"] for s in found_seeds if c in reports_data[s].index]
            
            per_class_rows.append({
                "Class": c,
                "P": calc_mean_std(p_vals),
                "R": calc_mean_std(r_vals),
                "F1": calc_mean_std(f_vals),
                "P_latex": calc_mean_std(p_vals, to_latex=True),
                "R_latex": calc_mean_std(r_vals, to_latex=True),
                "F1_latex": calc_mean_std(f_vals, to_latex=True)
            })
            
        df_per_class = pd.DataFrame(per_class_rows)
        print("\n" + "=" * 70)
        print("🌿 BẢNG ĐỘ ĐO P, R, F1 TỪNG LỚP THEO ĐỊNH DẠNG MEAN ± STD (10 lớp đầu):")
        print("=" * 70)
        display(df_per_class[["Class", "P", "R", "F1"]].head(10))

        # Lưu ra Excel & CSV
        out_excel = PROJECT / "Ket_Qua_Thuc_Nghiem_Mean_Std_Overleaf.xlsx"
        with pd.ExcelWriter(out_excel) as writer:
            summary_df.to_excel(writer, sheet_name="Tong_Quat", index=False)
            df_per_class.to_excel(writer, sheet_name="Tung_Loai_Cay", index=False)
        print(f"\n📁 Đã lưu file Excel: {out_excel}")

        # 3. TẠO MÃ NGUỒN LATEX SẴN SÀNG COPY VÀO OVERLEAF
        latex_code = []
        latex_code.append("% === COPY ĐOẠN NÀY DÁN THẲNG VÀO OVERLEAF ===")
        latex_code.append("\\begin{table}[htbp]")
        latex_code.append("  \\centering")
        latex_code.append("  \\caption{Kết quả đánh giá mô hình phân loại qua các Random Seed (Mean $\\pm$ Std)}")
        latex_code.append("  \\label{tab:seed_results}")
        latex_code.append("  \\begin{tabular}{l c c c}")
        latex_code.append("    \\toprule")
        latex_code.append("    \\textbf{Class / Metric} & \\textbf{P} & \\textbf{R} & \\textbf{F1} \\\\")
        latex_code.append("    \\midrule")
        
        # Thêm các chỉ số chính trước
        latex_code.append(f"    \\textbf{{Overall (Macro Avg)}} & {calc_mean_std(prec_list, to_latex=True)} & {calc_mean_std(rec_list, to_latex=True)} & {calc_mean_std(f1_list, to_latex=True)} \\\\")
        latex_code.append(f"    \\textbf{{Overall Accuracy}} & \\multicolumn{{3}}{{c}}{{{calc_mean_std(acc_list, to_latex=True)}}} \\\\")
        latex_code.append("    \\midrule")
        
        # Thêm 15 lớp tiêu biểu hoặc toàn bộ lớp
        for _, row in df_per_class.head(15).iterrows():
            clean_name = row['Class'].replace('_', ' ').replace('&', '\\&')
            latex_code.append(f"    {clean_name} & {row['P_latex']} & {row['R_latex']} & {row['F1_latex']} \\\\")
            
        latex_code.append("    \\bottomrule")
        latex_code.append("  \\end{tabular}")
        latex_code.append("\\end{table}")
        
        latex_str = "\n".join(latex_code)
        (PROJECT / "bang_ket_qua_overleaf.tex").write_text(latex_str, encoding="utf-8")
        
        print("\n" + "=" * 70)
        print("📄 MÃ LATEX DÁN VÀO OVERLEAF (Xem mẫu bên dưới, đã lưu vào bang_ket_qua_overleaf.tex):")
        print("=" * 70)
        print(latex_str)

